In [6]:
# IMPORTS

import pandas as pd
import numpy as np

In [7]:
# BUILD SIGNAL PANEL FUNCTION

def build_signal_panel(
    carry_path="data/carry_signal.parquet",
    trend_path="data/trend_signal.parquet",
    value_path="data/value_signal.parquet"
):
    # --- Load individual signal files ---
    carry = pd.read_parquet(carry_path)
    trend = pd.read_parquet(trend_path)
    value = pd.read_parquet(value_path)

    # --- Align indices ---
    common_index = carry.index.intersection(trend.index).intersection(value.index)

    carry = carry.loc[common_index]
    trend = trend.loc[common_index]
    value = value.loc[common_index]

    # --- Align columns/assets ---
    common_cols = carry.columns.intersection(trend.columns).intersection(value.columns)

    carry = carry[common_cols]
    trend = trend[common_cols]
    value = value[common_cols]

    # --- Add signal level to columns ---
    carry.columns = pd.MultiIndex.from_product([["carry"], carry.columns])
    trend.columns = pd.MultiIndex.from_product([["trend"], trend.columns])
    value.columns = pd.MultiIndex.from_product([["value"], value.columns])

    # --- Combine into one signal panel ---
    signal_panel = pd.concat([carry, trend, value], axis=1)

    # --- Sort for consistency ---
    signal_panel = signal_panel.sort_index(axis=1)

    return signal_panel

In [8]:
# BUILD SIGNAL PANEL
signal_panel = build_signal_panel()
signal_panel.columns.names = ["signal", "asset"]

# LOAD RETURNS
returns_df = pd.read_parquet("data/fx_returns_df_lagged.parquet")

# BASIC DIAGNOSTICS
print("signal_panel shape:", signal_panel.shape)
print("returns_df shape:", returns_df.shape)
print("signals:", signal_panel.columns.get_level_values("signal").unique().tolist())

signal_panel shape: (4524, 21)
returns_df shape: (5188, 7)
signals: ['carry', 'trend', 'value']


In [9]:
# Standalone sleeve performance
def performance_stats(pnl):
    pnl = pnl.dropna()

    ann = 252
    sharpe = pnl.mean() / pnl.std() * np.sqrt(ann) if pnl.std() != 0 else np.nan
    vol = pnl.std() * np.sqrt(ann)
    total_return = (1 + pnl).prod() - 1
    cum = (1 + pnl).cumprod()
    max_dd = (cum / cum.cummax() - 1).min()
    skew = pnl.skew()

    return {
        "PnL": total_return,
        "Sharpe": sharpe,
        "Vol": vol,
        "MaxDD": max_dd,
        "Skew": skew,
    }


def build_equal_weight_sleeve(signal_panel, returns_df, signal_name):
    sig = signal_panel[signal_name].copy()

    common_idx = sig.index.intersection(returns_df.index)
    common_cols = sig.columns.intersection(returns_df.columns)

    sig = sig.loc[common_idx, common_cols]
    rets = returns_df.loc[common_idx, common_cols]

    # cross-sectional normalization
    weights = sig.sub(sig.mean(axis=1), axis=0)
    gross = weights.abs().sum(axis=1)
    weights = weights.div(gross.replace(0, np.nan), axis=0).fillna(0)

    pnl = (weights.shift(1) * rets).sum(axis=1)

    return weights, pnl

def compute_turnover(weights):
    turnover = 0.5 * weights.diff().abs().sum(axis=1)
    return turnover

# CONFIG
COST_PER_TURNOVER = 0.0005  # 5 bps

# STORAGE
rows = []
sleeve_pnls = {}
sleeve_turnover = {}

# MAIN LOOP
for signal_name in signal_panel.columns.get_level_values("signal").unique():
    
    # build sleeve
    weights, pnl = build_equal_weight_sleeve(signal_panel, returns_df, signal_name)
    
    # turnover
    turnover = 0.5 * weights.diff().abs().sum(axis=1)
    
    # net pnl
    net_pnl = pnl - turnover * COST_PER_TURNOVER

    # store time series (IMPORTANT for later diagnostics)
    sleeve_pnls[signal_name] = pnl
    sleeve_turnover[signal_name] = turnover

    # stats
    gross_sharpe = performance_stats(pnl)["Sharpe"]
    net_sharpe = performance_stats(net_pnl)["Sharpe"]

    rows.append({
        "Sleeve": signal_name,
        "NetSharpe": net_sharpe,
        "GrossSharpe": gross_sharpe,
        "Turnover": turnover.mean() * 252  # annualized
    })

# FINAL TABLE
sleeve_stats = (
    pd.DataFrame(rows)
    .set_index("Sleeve")
    .sort_values("NetSharpe", ascending=False)
)

sleeve_stats

,NetSharpe,GrossSharpe,Turnover
Sleeve,,,
carry,0.483514,0.486538,0.367175
value,0.128010,0.148352,2.902792
trend,-0.241455,-0.111650,18.113827


In [10]:
# LOAD REGIME STATES

states_df = pd.read_parquet("data/states_df.parquet")

print("states_df shape:", states_df.shape)
print("states:", states_df.columns.tolist())

states_df shape: (9223, 4)
states: ['equity_state', 'commodity_state', 'vol_state', 'funding_state']


In [11]:
# SLEEVE PERFORMANCE BY NUMERIC STATE REGIME
def classify_state_regime(x, threshold=1.0):
    if pd.isna(x):
        return np.nan
    elif x >= threshold:
        return "high"
    elif x <= -threshold:
        return "low"
    else:
        return "neutral"


def sleeve_stats_by_numeric_state_regime(
    sleeve_pnls,
    states_path="data/states_df.parquet",
    threshold=1.0,
    min_obs=126
):
    states_df = pd.read_parquet(states_path)

    rows = []

    for sleeve_name, pnl in sleeve_pnls.items():
        pnl = pnl.dropna()

        common_idx = pnl.index.intersection(states_df.index)
        pnl = pnl.loc[common_idx]
        states = states_df.loc[common_idx]

        for state_name in states.columns:
            regime_series = states[state_name].apply(
                lambda x: classify_state_regime(x, threshold=threshold)
            )

            for regime in ["low", "neutral", "high"]:
                regime_idx = regime_series[regime_series == regime].index
                regime_pnl = pnl.loc[pnl.index.intersection(regime_idx)]

                if len(regime_pnl) < min_obs:
                    continue

                stats = performance_stats(regime_pnl)

                rows.append({
                    "Sleeve": sleeve_name,
                    "State": state_name,
                    "Regime": regime,
                    "Obs": len(regime_pnl),
                    "AvgStateValue": states.loc[regime_idx, state_name].mean(),
                    "PnL": stats["PnL"],
                    "Sharpe": stats["Sharpe"],
                    "Vol": stats["Vol"],
                    "MaxDD": stats["MaxDD"],
                    "Skew": stats["Skew"],
                })

    return pd.DataFrame(rows).sort_values(
        ["State", "Sleeve", "Regime"]
    ).reset_index(drop=True)


# RUN REGIME DIAGNOSTIC
sleeve_regime_stats = sleeve_stats_by_numeric_state_regime(
    sleeve_pnls=sleeve_pnls,
    states_path="data/states_df.parquet",
    threshold=1.0,
    min_obs=126
)

# PIVOT SHARPE FOR VISUAL READ
pivot = (
    sleeve_regime_stats
    .pivot_table(
        index=["State", "Sleeve"],
        columns="Regime",
        values="Sharpe"
    )
    .reindex(columns=["low", "neutral", "high"])
)

pivot

Regime                       low   neutral      high
State           Sleeve                              
commodity_state carry  -0.335136  0.115223  1.385841
                trend  -0.009042 -0.624041  0.046711
                value   0.249319  0.402415  0.025119
equity_state    carry  -1.406744 -0.191389  1.498131
                trend   0.212769 -0.104862 -0.946310
                value  -0.034282 -0.051879  1.087364
funding_state   carry   3.626277  0.639489 -1.851555
                trend  -0.839927  0.051714 -0.870772
                value   0.471189  0.001565  1.060275
vol_state       carry   0.937935  0.346776  0.096397
                trend   0.158281 -0.622707  0.288006
                value  -0.105120  0.563638 -0.070948

In [ ]:
# BUILD IC-BASED SLEEVE SCALARS (EXPANDING WINDOW)
def build_ic_sleeve_scalars_expanding(
    sleeve_pnls,
    states_df,
    lookback=504,
    min_obs=126,
    strength=0.25,
    max_scalar=1.50,
    min_scalar=0.50,
    smooth_window=21
):

    signal_names = list(sleeve_pnls.keys())
    scalar_df = pd.DataFrame(index=states_df.index, columns=signal_names, dtype=float)

    dates = states_df.index

    for t in range(lookback, len(dates)):
        current_date = dates[t]
        window_idx = dates[t - lookback:t]

        scalar_inputs = {}

        for sleeve_name, pnl in sleeve_pnls.items():
            pnl_window = pnl.loc[pnl.index.intersection(window_idx)]

            if len(pnl_window) < min_obs:
                continue

            state_effect = 0.0

            for state in states_df.columns:
                state_window = states_df[state].loc[
                    states_df.index.intersection(pnl_window.index)
                ]

                common_idx = pnl_window.index.intersection(state_window.index)

                if len(common_idx) < min_obs:
                    continue

                x = state_window.loc[common_idx]
                y = pnl_window.loc[common_idx]

                if x.std() == 0 or y.std() == 0:
                    continue

                ic = x.corr(y)

                if pd.isna(ic):
                    continue

                state_effect += ic * states_df.loc[current_date, state]

            scalar_inputs[sleeve_name] = state_effect

        scalar_signal = pd.Series(scalar_inputs)

        # CONVEX TRANSFORM
        p = 1.5
        cap = 3.0

        convex_signal = np.sign(scalar_signal) * (np.abs(scalar_signal) ** p)
        convex_signal = convex_signal.clip(-cap, cap)

        scalar = np.exp(strength * convex_signal)
        scalar = scalar.clip(lower=min_scalar, upper=max_scalar)

        # CRITICAL LINE
        scalar_df.loc[current_date, scalar.index] = scalar

    scalar_df = scalar_df.rolling(smooth_window, min_periods=1).mean()
    scalar_df = scalar_df.shift(1)

    return scalar_df

sleeve_scalars = build_ic_sleeve_scalars_expanding(
    sleeve_pnls=sleeve_pnls,
    states_df=states_df,
    lookback=504,
    min_obs=126,
    strength=0.25,
    max_scalar=1.50,
    min_scalar=0.50,
    smooth_window=21
)

/opt/anaconda3/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:2991: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/opt/anaconda3/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:2848: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/opt/anaconda3/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:2848: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
/opt/anaconda3/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:2991: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/opt/anaconda3/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:2848: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/opt/anaconda3/lib/python3.13/site-packages/numpy/lib/_function_base_impl.py:2848: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide

In [18]:
# EQUAL WEIGHT COMBINED SIGNAL PORTFOLIO
def build_equal_signal_combo_pnl(signal_panel, returns_df, cost_per_turnover=0.0005):
    # --- Get signal names ---
    signal_names = signal_panel.columns.get_level_values("signal").unique()

    # --- Equal-weight combine raw signals (FIXED) ---
    combined_signal = (
        signal_panel
        .T
        .groupby(level="asset")
        .mean()
        .T
    )

    # --- Align with returns ---
    common_idx = combined_signal.index.intersection(returns_df.index)
    common_cols = combined_signal.columns.intersection(returns_df.columns)

    combined_signal = combined_signal.loc[common_idx, common_cols]
    rets = returns_df.loc[common_idx, common_cols]

    # --- Demean cross-sectionally for dollar neutrality ---
    weights = combined_signal.sub(combined_signal.mean(axis=1), axis=0)

    # --- Gross normalize to 1.0 ---
    gross = weights.abs().sum(axis=1)
    weights = weights.div(gross.replace(0, np.nan), axis=0).fillna(0)

    # --- One-day execution lag ---
    pnl = (weights.shift(1) * rets).sum(axis=1)

    # --- Compute turnover (0.5 convention) ---
    turnover = compute_turnover(weights)

    # --- Apply transaction costs ---
    net_pnl = pnl - turnover * cost_per_turnover

    return pnl, net_pnl, weights, turnover

combo_pnl, combo_net_pnl, combo_weights, combo_turnover = build_equal_signal_combo_pnl(
    signal_panel=signal_panel,
    returns_df=returns_df
)

combo_stats = pd.DataFrame({
    "GrossSharpe": performance_stats(combo_pnl)["Sharpe"],
    "NetSharpe": performance_stats(combo_net_pnl)["Sharpe"],
    "Turnover": combo_turnover.mean() * 252
}, index=["equal_signal_combo"])

combo_stats

,GrossSharpe,NetSharpe,Turnover
equal_signal_combo,0.368655,0.241609,14.831765


In [19]:
# INVERSE VOL SLEEVE COMBO PORTFOLIO
def build_inverse_vol_sleeve_combo_pnl(
    signal_panel,
    returns_df,
    vol_window=60,
    vol_floor=0.01,
    cost_per_turnover=0.0005
):
    sleeve_pnls = {}
    sleeve_weights = {}

    signal_names = signal_panel.columns.get_level_values("signal").unique()

    # --- Build standalone sleeve weights and PnLs ---
    for signal_name in signal_names:
        sig = signal_panel[signal_name].copy()

        common_idx = sig.index.intersection(returns_df.index)
        common_cols = sig.columns.intersection(returns_df.columns)

        sig = sig.loc[common_idx, common_cols]
        rets = returns_df.loc[common_idx, common_cols]

        weights = sig.sub(sig.mean(axis=1), axis=0)
        gross = weights.abs().sum(axis=1)
        weights = weights.div(gross.replace(0, np.nan), axis=0).fillna(0)

        pnl = (weights.shift(1) * rets).sum(axis=1)

        sleeve_weights[signal_name] = weights
        sleeve_pnls[signal_name] = pnl

    sleeve_pnls_df = pd.DataFrame(sleeve_pnls)

    # --- Lagged inverse vol sleeve allocation ---
    trailing_vol = sleeve_pnls_df.rolling(vol_window).std() * np.sqrt(252)
    trailing_vol = trailing_vol.shift(1)

    inv_vol = 1 / trailing_vol.clip(lower=vol_floor)
    sleeve_alloc = inv_vol.div(inv_vol.sum(axis=1), axis=0).fillna(0)

    # --- Combine asset weights using sleeve allocations ---
    final_weights = pd.DataFrame(0.0, index=returns_df.index, columns=returns_df.columns)

    for signal_name in signal_names:
        w = sleeve_weights[signal_name]

        common_idx = final_weights.index.intersection(w.index).intersection(sleeve_alloc.index)
        common_cols = final_weights.columns.intersection(w.columns)

        alloc = sleeve_alloc.loc[common_idx, signal_name]
        final_weights.loc[common_idx, common_cols] += w.loc[common_idx, common_cols].mul(alloc, axis=0)

    # --- Final gross normalize for tradability ---
    gross = final_weights.abs().sum(axis=1)
    final_weights = final_weights.div(gross.replace(0, np.nan), axis=0).fillna(0)

    # --- One-day execution lag ---
    common_idx = final_weights.index.intersection(returns_df.index)
    common_cols = final_weights.columns.intersection(returns_df.columns)

    pnl = (
        final_weights.loc[common_idx, common_cols].shift(1)
        * returns_df.loc[common_idx, common_cols]
    ).sum(axis=1)

    # --- Compute turnover at portfolio level ---
    turnover = compute_turnover(final_weights)

    # --- Apply transaction costs ---
    net_pnl = pnl - turnover * cost_per_turnover

    return pnl, net_pnl, final_weights, turnover, sleeve_alloc, sleeve_pnls_df

ivol_combo_pnl, ivol_combo_net_pnl, ivol_combo_weights, ivol_turnover, ivol_sleeve_alloc, sleeve_pnls_df = (
    build_inverse_vol_sleeve_combo_pnl(
        signal_panel=signal_panel,
        returns_df=returns_df,
        vol_window=60,
        vol_floor=0.01
    )
)

ivol_combo_stats = pd.DataFrame({
    "GrossSharpe": performance_stats(ivol_combo_pnl)["Sharpe"],
    "NetSharpe": performance_stats(ivol_combo_net_pnl)["Sharpe"],
    "Turnover": ivol_turnover.mean() * 252
}, index=["inverse_vol_sleeve_combo"])

ivol_combo_stats

,GrossSharpe,NetSharpe,Turnover
inverse_vol_sleeve_combo,0.43865,0.33127,11.895845


In [20]:
# SCALAR WEIGHTED SLEEVE COMBO PORTFOLIO
def build_scalar_weighted_sleeve_combo_pnl(
    signal_panel,
    returns_df,
    sleeve_scalars,
    cost_per_turnover=0.0005
):
    sleeve_pnls = {}
    sleeve_weights = {}

    signal_names = signal_panel.columns.get_level_values("signal").unique()

    # --- Build standalone sleeve weights and PnLs ---
    for signal_name in signal_names:
        sig = signal_panel[signal_name].copy()

        common_idx = sig.index.intersection(returns_df.index)
        common_cols = sig.columns.intersection(returns_df.columns)

        sig = sig.loc[common_idx, common_cols]
        rets = returns_df.loc[common_idx, common_cols]

        weights = sig.sub(sig.mean(axis=1), axis=0)

        gross = weights.abs().sum(axis=1)
        weights = weights.div(gross.replace(0, np.nan), axis=0).fillna(0)

        pnl = (weights.shift(1) * rets).sum(axis=1)

        sleeve_weights[signal_name] = weights
        sleeve_pnls[signal_name] = pnl

    sleeve_pnls_df = pd.DataFrame(sleeve_pnls)

    # --- Use sleeve scalars as dynamic sleeve allocation ---
    sleeve_alloc = sleeve_scalars.copy()

    common_idx = sleeve_pnls_df.index.intersection(sleeve_alloc.index)
    common_cols = sleeve_pnls_df.columns.intersection(sleeve_alloc.columns)

    sleeve_alloc = sleeve_alloc.loc[common_idx, common_cols]

    # --- Renormalize scalars into sleeve weights ---
    sleeve_alloc = sleeve_alloc.div(
        sleeve_alloc.sum(axis=1).replace(0, np.nan),
        axis=0
    ).fillna(0)

    # --- Combine asset weights using scalar-based sleeve allocations ---
    final_weights = pd.DataFrame(
        0.0,
        index=returns_df.index,
        columns=returns_df.columns
    )

    for signal_name in common_cols:
        w = sleeve_weights[signal_name]

        idx = final_weights.index.intersection(w.index).intersection(sleeve_alloc.index)
        cols = final_weights.columns.intersection(w.columns)

        alloc = sleeve_alloc.loc[idx, signal_name]

        final_weights.loc[idx, cols] += w.loc[idx, cols].mul(alloc, axis=0)

    # --- Final gross normalize for tradability ---
    gross = final_weights.abs().sum(axis=1)
    final_weights = final_weights.div(gross.replace(0, np.nan), axis=0).fillna(0)

    # --- One-day execution lag ---
    common_idx = final_weights.index.intersection(returns_df.index)
    common_cols = final_weights.columns.intersection(returns_df.columns)

    pnl = (
        final_weights.loc[common_idx, common_cols].shift(1)
        * returns_df.loc[common_idx, common_cols]
    ).sum(axis=1)

    # --- Compute turnover at portfolio level ---
    turnover = compute_turnover(final_weights)

    # --- Apply transaction costs ---
    net_pnl = pnl - turnover * cost_per_turnover

    return pnl, net_pnl, final_weights, turnover, sleeve_alloc, sleeve_pnls_df

scalar_combo_pnl, scalar_combo_net_pnl, scalar_combo_weights, scalar_turnover, scalar_sleeve_alloc, scalar_sleeve_pnls_df = (
    build_scalar_weighted_sleeve_combo_pnl(
        signal_panel=signal_panel,
        returns_df=returns_df,
        sleeve_scalars=sleeve_scalars
    )
)

scalar_combo_stats = pd.DataFrame({
    "GrossSharpe": performance_stats(scalar_combo_pnl)["Sharpe"],
    "NetSharpe": performance_stats(scalar_combo_net_pnl)["Sharpe"],
    "Turnover": scalar_turnover.mean() * 252
}, index=["scalar_weighted_sleeve_combo"])

scalar_combo_stats

,GrossSharpe,NetSharpe,Turnover
scalar_weighted_sleeve_combo,0.310942,0.188842,13.562144


In [21]:
# INVERSE VOL × SCALAR OVERLAY SLEEVE COMBO PORTFOLIO
def build_inverse_vol_scalar_overlay_combo_pnl(
    signal_panel,
    returns_df,
    sleeve_scalars,
    vol_window=60,
    vol_floor=0.01,
    cost_per_turnover=0.0005
):
    sleeve_pnls = {}
    sleeve_weights = {}

    signal_names = signal_panel.columns.get_level_values("signal").unique()

    # --- Build standalone sleeve weights and PnLs ---
    for signal_name in signal_names:
        sig = signal_panel[signal_name].copy()

        common_idx = sig.index.intersection(returns_df.index)
        common_cols = sig.columns.intersection(returns_df.columns)

        sig = sig.loc[common_idx, common_cols]
        rets = returns_df.loc[common_idx, common_cols]

        weights = sig.sub(sig.mean(axis=1), axis=0)
        gross = weights.abs().sum(axis=1)
        weights = weights.div(gross.replace(0, np.nan), axis=0).fillna(0)

        pnl = (weights.shift(1) * rets).sum(axis=1)

        sleeve_weights[signal_name] = weights
        sleeve_pnls[signal_name] = pnl

    sleeve_pnls_df = pd.DataFrame(sleeve_pnls)

    # --- Base sleeve allocation: lagged inverse vol ---
    trailing_vol = sleeve_pnls_df.rolling(vol_window).std() * np.sqrt(252)
    trailing_vol = trailing_vol.shift(1)

    inv_vol = 1 / trailing_vol.clip(lower=vol_floor)
    base_sleeve_alloc = inv_vol.div(inv_vol.sum(axis=1), axis=0).fillna(0)

    # --- Align scalar overlay ---
    common_idx = base_sleeve_alloc.index.intersection(sleeve_scalars.index)
    common_cols = base_sleeve_alloc.columns.intersection(sleeve_scalars.columns)

    base_sleeve_alloc = base_sleeve_alloc.loc[common_idx, common_cols]
    aligned_scalars = sleeve_scalars.loc[common_idx, common_cols]

    # --- Apply scalar overlay ---
    sleeve_alloc = base_sleeve_alloc * aligned_scalars

    # --- Renormalize sleeve allocation ---
    sleeve_alloc = sleeve_alloc.div(
        sleeve_alloc.sum(axis=1).replace(0, np.nan),
        axis=0
    ).fillna(0)

    # --- Combine asset weights ---
    final_weights = pd.DataFrame(
        0.0,
        index=returns_df.index,
        columns=returns_df.columns
    )

    for signal_name in common_cols:
        w = sleeve_weights[signal_name]

        idx = final_weights.index.intersection(w.index).intersection(sleeve_alloc.index)
        cols = final_weights.columns.intersection(w.columns)

        alloc = sleeve_alloc.loc[idx, signal_name]
        final_weights.loc[idx, cols] += w.loc[idx, cols].mul(alloc, axis=0)

    # --- Final gross normalize for tradability ---
    gross = final_weights.abs().sum(axis=1)
    final_weights = final_weights.div(gross.replace(0, np.nan), axis=0).fillna(0)

    # --- One-day execution lag ---
    common_idx = final_weights.index.intersection(returns_df.index)
    common_cols = final_weights.columns.intersection(returns_df.columns)

    pnl = (
        final_weights.loc[common_idx, common_cols].shift(1)
        * returns_df.loc[common_idx, common_cols]
    ).sum(axis=1)

    # --- Compute turnover at portfolio level ---
    turnover = compute_turnover(final_weights)

    # --- Apply transaction costs ---
    net_pnl = pnl - turnover * cost_per_turnover

    return pnl, net_pnl, final_weights, turnover, sleeve_alloc, base_sleeve_alloc, sleeve_pnls_df

ivol_scalar_pnl, ivol_scalar_net_pnl, ivol_scalar_weights, ivol_scalar_turnover, ivol_scalar_alloc, base_ivol_alloc, sleeve_pnls_df = (
    build_inverse_vol_scalar_overlay_combo_pnl(
        signal_panel=signal_panel,
        returns_df=returns_df,
        sleeve_scalars=sleeve_scalars,
        vol_window=60,
        vol_floor=0.01
    )
)

ivol_scalar_stats = pd.DataFrame({
    "GrossSharpe": performance_stats(ivol_scalar_pnl)["Sharpe"],
    "NetSharpe": performance_stats(ivol_scalar_net_pnl)["Sharpe"],
    "Turnover": ivol_scalar_turnover.mean() * 252
}, index=["inverse_vol_scalar_overlay_combo"])

ivol_scalar_stats

,GrossSharpe,NetSharpe,Turnover
inverse_vol_scalar_overlay_combo,0.398659,0.288972,12.151374


In [22]:
# COLLECT PORTFOLIO DIAGNOSTICS AND SAVE
def collect_portfolio_diagnostics(
    portfolio_name,
    pnl,
    net_pnl,
    weights,
    turnover,
    returns_df,
    sleeve_alloc=None,
    sleeve_pnls_df=None,
    save_dir="data/diagnostics"
):
    """
    Collects and saves all relevant portfolio construction outputs.

    Saves:
    - pnl (gross and net)
    - weights
    - turnover
    - returns alignment
    - optional sleeve allocations
    - optional sleeve pnl panel
    - summary stats

    All saved as parquet for downstream analysis.
    """

    import os

    os.makedirs(save_dir, exist_ok=True)

    # --- Align everything ---
    common_idx = pnl.index.intersection(net_pnl.index).intersection(weights.index)
    pnl = pnl.loc[common_idx]
    net_pnl = net_pnl.loc[common_idx]
    weights = weights.loc[common_idx]
    turnover = turnover.loc[common_idx]

    # --- Combine pnl ---
    pnl_df = pd.DataFrame({
        "gross": pnl,
        "net": net_pnl
    })

    # --- Summary stats ---
    stats = {
        "GrossSharpe": performance_stats(pnl)["Sharpe"],
        "NetSharpe": performance_stats(net_pnl)["Sharpe"],
        "Vol": performance_stats(net_pnl)["Vol"],
        "MaxDD": performance_stats(net_pnl)["MaxDD"],
        "Turnover": turnover.mean() * 252
    }

    stats_df = pd.DataFrame(stats, index=[portfolio_name])

    # --- Save core objects ---
    pnl_df.to_parquet(f"{save_dir}/{portfolio_name}_pnl.parquet")
    weights.to_parquet(f"{save_dir}/{portfolio_name}_weights.parquet")
    turnover.to_frame("turnover").to_parquet(f"{save_dir}/{portfolio_name}_turnover.parquet")
    stats_df.to_parquet(f"{save_dir}/{portfolio_name}_stats.parquet")

    # --- Save optional objects ---
    if sleeve_alloc is not None:
        sleeve_alloc.to_parquet(f"{save_dir}/{portfolio_name}_sleeve_alloc.parquet")

    if sleeve_pnls_df is not None:
        sleeve_pnls_df.to_parquet(f"{save_dir}/{portfolio_name}_sleeve_pnls.parquet")

    # --- Save returns for downstream alignment ---
    aligned_returns = returns_df.loc[common_idx]
    aligned_returns.to_parquet(f"{save_dir}/{portfolio_name}_returns.parquet")

    return {
        "pnl": pnl_df,
        "weights": weights,
        "turnover": turnover,
        "stats": stats_df
    }

diag = collect_portfolio_diagnostics(
    portfolio_name="ivol_scalar_overlay",
    pnl=ivol_scalar_pnl,
    net_pnl=ivol_scalar_net_pnl,
    weights=ivol_scalar_weights,
    turnover=ivol_scalar_turnover,
    returns_df=returns_df,
    sleeve_alloc=ivol_scalar_alloc,
    sleeve_pnls_df=sleeve_pnls_df
)

print("done")

done
